# Deep Learning para Predicción de Demanda
## Redes Neuronales Avanzadas con TensorFlow/Keras

En este notebook implementamos redes neuronales profundas para predecir la demanda de productos en un sistema de retail.

## 1. Carga de librerías y configuración del entorno

In [ ]:
# Importar librerías
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# TensorFlow y Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Configurar semilla para reproducibilidad
np.random.seed(42)
tf.random.set_seed(42)

# Verificar GPU
print(f"GPU disponible: {tf.config.list_physical_devices('GPU')}")
print(f"Versión de TensorFlow: {tf.__version__}")

# Configurar matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## 2. Carga y exploración del dataset

In [ ]:
# Cargar datos
ruta_datos = '../../../Retail_Sales_Data.csv'
df = pd.read_csv(ruta_datos)

# Exploración inicial
print("=== INFORMACIÓN DEL DATASET ===")
print(f"Dimensiones: {df.shape}")
print(f"\nPrimeras filas:")
print(df.head())
print(f"\nTipos de datos:")
print(df.dtypes)
print(f"\nValores nulos:")
print(df.isnull().sum())
print(f"\nEstadísticas descriptivas:")
print(df.describe())

## 3. Preprocesamiento y partición de datos

In [ ]:
# Preparar datos
df['Date_of_Sale'] = pd.to_datetime(df['Date_of_Sale'])

# Ingeniería de características (feature engineering)
df['Mes'] = df['Date_of_Sale'].dt.month
df['DiaSemana'] = df['Date_of_Sale'].dt.dayofweek
df['Trimestre'] = df['Date_of_Sale'].dt.quarter

# Agrupar por fecha y categoría para obtener demanda
df_agrupado = df.groupby(['Date_of_Sale', 'Product_Category', 'Mes', 'DiaSemana', 'Trimestre'])['Sales_Amount'].sum().reset_index()
df_agrupado.rename(columns={'Sales_Amount': 'DemandaTotal'}, inplace=True)

print(f"Dataset agrupado: {df_agrupado.shape}")
print(f"Características temporales añadidas: Mes, DiaSemana, Trimestre")

# Codificar categoría de producto
le = LabelEncoder()
df_agrupado['Categoria_Codificada'] = le.fit_transform(df_agrupado['Product_Category'])

# Seleccionar características
caracteristicas = ['Mes', 'DiaSemana', 'Trimestre', 'Categoria_Codificada']
X = df_agrupado[caracteristicas].values
y = df_agrupado['DemandaTotal'].values

# Normalizar datos (CRUCIAL para redes neuronales)
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_escalado = scaler_X.fit_transform(X)
y_escalado = scaler_y.fit_transform(y.reshape(-1, 1)).flatten()

# Dividir en entrenamiento, validación y prueba
X_temp, X_prueba, y_temp, y_prueba = train_test_split(X_escalado, y_escalado, test_size=0.1, random_state=42)
X_entrenamiento, X_validacion, y_entrenamiento, y_validacion = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42)

print(f"\nTamaños de conjuntos:")
print(f"Entrenamiento: {X_entrenamiento.shape[0]} muestras")
print(f"Validación: {X_validacion.shape[0]} muestras")
print(f"Prueba: {X_prueba.shape[0]} muestras")

## 4. Definición de la red neuronal profunda (Keras)

In [ ]:
# Construir Red Neuronal Profunda
print("=== CONSTRUCCIÓN DE RED NEURONAL DENSA ===\n")

modelo_densa = models.Sequential([
    # Primera capa oculta
    layers.Dense(128, activation='relu', input_shape=(X_entrenamiento.shape[1],),
                 kernel_regularizer=keras.regularizers.l2(0.01)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    
    # Segunda capa oculta
    layers.Dense(64, activation='relu',
                 kernel_regularizer=keras.regularizers.l2(0.01)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    
    # Tercera capa oculta
    layers.Dense(32, activation='relu',
                 kernel_regularizer=keras.regularizers.l2(0.01)),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    
    # Cuarta capa oculta
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.2),
    
    # Capa de salida (regresión)
    layers.Dense(1, activation='linear')
])

print("Arquitectura del modelo:")
modelo_densa.summary()

## 5. Compilación del modelo y métricas

In [ ]:
# Compilar modelo con Adam optimizer
optimizer = Adam(learning_rate=0.001)

modelo_densa.compile(
    optimizer=optimizer,
    loss='mse',  # Mean Squared Error para regresión
    metrics=['mae', 'mse']  # MAE y MSE como métricas
)

print("✓ Modelo compilado exitosamente")
print(f"Optimizador: Adam (learning_rate=0.001)")
print(f"Función de pérdida: MSE")
print(f"Métricas: MAE, MSE")

## 6. Entrenamiento con validación y callbacks

In [ ]:
# Definir callbacks para mejorar el entrenamiento
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-7,
    verbose=1
)

# Entrenar modelo
print("=== INICIANDO ENTRENAMIENTO ===\n")
historial = modelo_densa.fit(
    X_entrenamiento, y_entrenamiento,
    epochs=100,
    batch_size=32,
    validation_data=(X_validacion, y_validacion),
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

print("\n✓ Entrenamiento completado")

In [ ]:
# Visualizar historial de entrenamiento
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Gráfico de pérdida
axes[0].plot(historial.history['loss'], label='Pérdida Entrenamiento', linewidth=2)
axes[0].plot(historial.history['val_loss'], label='Pérdida Validación', linewidth=2)
axes[0].set_title('Pérdida por Época', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Pérdida (MSE)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Gráfico de MAE
axes[1].plot(historial.history['mae'], label='MAE Entrenamiento', linewidth=2)
axes[1].plot(historial.history['val_mae'], label='MAE Validación', linewidth=2)
axes[1].set_title('Error Absoluto Medio por Época', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Gráficos de historial generados")

## 7. Evaluación en conjunto de prueba

In [ ]:
# Evaluar modelo en conjunto de prueba
print("=== EVALUACIÓN EN CONJUNTO DE PRUEBA ===\n")

resultados_prueba = modelo_densa.evaluate(X_prueba, y_prueba, verbose=0)

print(f"Pérdida (MSE): {resultados_prueba[0]:.6f}")
print(f"MAE: {resultados_prueba[1]:.6f}")
print(f"MSE: {resultados_prueba[2]:.6f}")

# Realizar predicciones
y_pred_escalado = modelo_densa.predict(X_prueba, verbose=0)
y_pred = scaler_y.inverse_transform(y_pred_escalado).flatten()
y_prueba_original = scaler_y.inverse_transform(y_prueba.reshape(-1, 1)).flatten()

# Calcular métricas adicionales
mse = mean_squared_error(y_prueba_original, y_pred)
mae = mean_absolute_error(y_prueba_original, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_prueba_original, y_pred)

print(f"\nMétricas en escala original:")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R² Score: {r2:.4f}")